<a href="https://colab.research.google.com/github/mena-04/DoS-Stress-Testing/blob/main/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# background metrics sampler

In [5]:
import threading
import requests
import time
import re
import csv

stop_sampling = False
samples = []

def get_value(text, name):
    pattern = rf'^{re.escape(name)}\{{.*?\}}\s+([0-9.eE+-]+)'
    m = re.search(pattern, text, re.MULTILINE)
    return float(m.group(1)) if m else None

def sampler():
   global sample_errors
   sample_errors = 0
   while not stop_sampling:
        try:
            text = requests.get(
                "http://127.0.0.1:8000/metrics",
                timeout=2
            ).text

            samples.append({
                "timestamp": time.time(),
                "running": get_value(
                    text,
                    "vllm:num_requests_running"
                ),
                "waiting": get_value(
                    text,
                    "vllm:num_requests_waiting"
                )
            })
        except Exception:
            pass

        time.sleep(0.25)

thread = threading.Thread(target=sampler, daemon=True)
thread.start()

print("sampler started")

sampler started


# concurrent overload test

In [6]:
import asyncio
import aiohttp
import time
import statistics

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30

async def send_one(session, i):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": EXPENSIVE_PROMPT}
        ],
        "max_tokens": 128,
        "temperature": 0
    }

    start = time.perf_counter()

    try:
        async with session.post(URL, json=payload, timeout=120) as r:
            await r.text()
            latency = time.perf_counter() - start
            return {
                "id": i,
                "status": r.status,
                "latency": latency
            }
    except Exception as e:
        return {
            "id": i,
            "status": "error",
            "latency": time.perf_counter() - start
        }

async def run_load(n=50):
    async with aiohttp.ClientSession() as session:
        tasks = [
            asyncio.create_task(send_one(session, i))
            for i in range(n)
        ]
        return await asyncio.gather(*tasks)

results = await run_load(50)

latencies = [
    r["latency"]
    for r in results
    if r["status"] == 200
]

print("completed:", len(results))
print("successful:", len(latencies))

if latencies:
    print("p50:", round(statistics.median(latencies), 3))

    sorted_lat = sorted(latencies)
    p95_index = int(0.95 * len(sorted_lat)) - 1
    print("p95:", round(sorted_lat[p95_index], 3))

    print("max:", round(max(latencies), 3))

completed: 50
successful: 0


In [7]:
stop_sampling = True
thread.join(timeout=2)

print("samples:", len(samples))
print("max running:", max(x["running"] or 0 for x in samples))
print("max waiting:", max(x["waiting"] or 0 for x in samples))

samples: 0


ValueError: max() iterable argument is empty

# Mitigation Gateway

In [8]:
# Clone the repository
%cd /content
!rm -rf DoS-Stress-Testing
!git clone https://github.com/mena-04/DoS-Stress-Testing.git
%cd /content/DoS-Stress-Testing

/content
Cloning into 'DoS-Stress-Testing'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 150 (delta 47), reused 120 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 186.78 KiB | 10.99 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/DoS-Stress-Testing


In [9]:
import subprocess, time, requests, sys

RUN_ID = "mitigated-01"     # change per run; names the output folder

gw = subprocess.Popen([
    sys.executable, "-m", "gateway",
    "--config", "configs/ratelimit_queue.yaml",
    "--upstream", "http://127.0.0.1:8000",
    "--port", "8080",
    "--run-id", RUN_ID,
    "--log-dir", "/content/runs",
], stdout=open("/content/gateway.log", "w"), stderr=subprocess.STDOUT,
   cwd="/content/DoS-Stress-Testing")

print("gateway PID:", gw.pid)

for _ in range(30):
    if gw.poll() is not None:
        print("GATEWAY EXITED, code:", gw.returncode)
        break
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200:
            print("GATEWAY READY")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print("timed out")

gateway PID: 2799
GATEWAY EXITED, code: 3


In [10]:
!curl -s -o /dev/null -w "%{http_code}" localhost:8000/health

000